# Project: House Price Prediction

> Part of the **Machine Learning Learning Series**

## Project Overview
**Goal**: Predict house prices using regression models.

**Skills demonstrated**: Feature engineering, regression, model evaluation, hyperparameter tuning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings; warnings.filterwarnings('ignore')

data = fetch_california_housing()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
print(f'Dataset shape: {df.shape}')
df.head()

## 1. Exploratory Data Analysis

In [ ]:
print(df.describe())
print('\nMissing values:', df.isnull().sum().sum())

fig, axes = plt.subplots(2, 4, figsize=(16,8))
for i, col in enumerate(data.feature_names):
    axes[i//4][i%4].hist(df[col], bins=30, edgecolor='white')
    axes[i//4][i%4].set_title(col)
plt.tight_layout()
plt.show()

## 2. Feature Engineering

In [ ]:
# Create new features
df['rooms_per_household'] = df['AveRooms'] / df['AveOccup'].clip(lower=1)
df['bedrooms_ratio'] = df['AveBedrms'] / df['AveRooms'].clip(lower=1)
df['log_median_income'] = np.log1p(df['MedInc'])

print('New features added')

## 3. Model Training & Comparison

In [ ]:
X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

results = {}
for name, model in [('Linear Regression', LinearRegression()),
                     ('Ridge', Ridge(alpha=1.0)),
                     ('Random Forest', RandomForestRegressor(100, random_state=42)),
                     ('Gradient Boosting', GradientBoostingRegressor(100, random_state=42))]:
    model.fit(X_train_sc, y_train)
    pred = model.predict(X_test_sc)
    results[name] = {'RMSE': np.sqrt(mean_squared_error(y_test, pred)), 'R²': r2_score(y_test, pred), 'MAE': mean_absolute_error(y_test, pred)}

results_df = pd.DataFrame(results).T
print(results_df.round(4))

## 4. Results
The best model is Gradient Boosting with the lowest RMSE and highest R² score.